# 03 — Clean invalid observations; preserve possible genuine surges
Remove invalid/non-positive prices and wholesale records. Do **not** delete items using
full-training coverage: Notebook 06 learns exclusions separately in each fold.

IQR flags are diagnostic only, never features or automatic deletions. All valid positive
prices are retained, including the six previously suspected tenfold errors. Their causes
were not externally confirmed. This avoids undocumented corrections and manual exclusions
that change when different months are examined. Investigate flagged records and acknowledge
residual label uncertainty; genuine extreme prices must not automatically disappear.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Run from the repository root or its notebooks folder.")
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import numpy as np
import pricecatcher_config as cfg
from data_utils import save_json, surge_event
output = cfg.stage(3)
df = pd.read_parquet(cfg.stage(1) / "integrated.parquet")
df["date"] = pd.to_datetime(df["date"], errors="raise")
if df[["date", "premise_code"] + cfg.SERIES_KEYS].isna().any().any():
    raise ValueError("Missing date or series key.")
if df.duplicated(["date", "premise_code", "item_code"]).any():
    raise ValueError("Duplicate keys need review, not arbitrary keep-first removal.")
original_rows = len(df)
invalid = df["price"].isna() | ~np.isfinite(df["price"]) | df["price"].le(0)
invalid_count = int(invalid.sum())
valid = df.loc[~invalid].copy()
wholesale = valid["premise_type"].eq("Borong") if cfg.REMOVE_BORONG else pd.Series(False, index=valid.index)
wholesale_count = int(wholesale.sum())
clean = valid.loc[~wholesale].copy()
reference = clean.loc[clean["date"] <= pd.Timestamp(cfg.TRAIN_END)]
if reference.empty:
    raise ValueError("No training records for diagnostic bounds.")
bounds = reference.groupby("item_code")["price"].agg(
    median="median", q1=lambda x: x.quantile(0.25), q3=lambda x: x.quantile(0.75))
iqr = bounds["q3"] - bounds["q1"]
bounds["lower_bound"] = (bounds["q1"] - 3 * iqr).clip(lower=0)
bounds["upper_bound"] = bounds["q3"] + 3 * iqr
zero = iqr.eq(0)
bounds.loc[zero, "lower_bound"] = bounds.loc[zero, "median"] * 0.5
bounds.loc[zero, "upper_bound"] = bounds.loc[zero, "median"] * 2
clean = clean.merge(bounds[["lower_bound", "upper_bound"]], on="item_code", how="left", validate="many_to_one")
clean["is_suspicious_price"] = clean["price"].lt(clean["lower_bound"]) | clean["price"].gt(clean["upper_bound"])
review = clean.loc[clean["is_suspicious_price"] & clean["date"].le(pd.Timestamp(cfg.TRAIN_END))]
review.to_csv(output / "training_price_review.csv", index=False)
bounds.to_csv(output / "diagnostic_price_bounds.csv")
clean.drop(columns=["lower_bound", "upper_bound"]).to_parquet(output / "daily.parquet", index=False)
save_json(output / "cleaning_summary.json", {
    "original_rows": original_rows, "invalid_removed": invalid_count,
    "wholesale_removed": wholesale_count, "retained_rows": len(clean),
    "flagged_retained": int(clean["is_suspicious_price"].sum()),
    "policy": "all valid positive retail prices retained; diagnostic flags excluded from model features",
})
print("Saved:", len(clean), "daily records; removed invalid:", invalid_count, "wholesale:", wholesale_count)
display(review.head(10))
